# r2-001 Problem 4 — Reference solution (texture patches, 5% labels)

One reference approach: fixed, label-free texture features (pixel statistics, a rotation-invariant Fourier power summary, and neighbour correlations), standardized on the labelled + unlabelled pool, a multinomial logistic-regression head trained on the 40 labels, then confidence-thresholded self-training on the 760-patch pool.
The cell below is the problem notebook's supplied setup cell, unchanged.

In [ ]:
import importlib.util
import json
import math
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261101
torch.set_num_threads(1)


def find_data_dir() -> Path:
    """Locate r2-001's data/ folder from the notebook's working directory."""
    candidates = [Path("../data"), Path("data"), Path("mocktests/r2-001/data"), Path("book2/mocktests/r2-001/data")]
    env_root = os.environ.get("USAAIO_BOOK_ROOT")
    if env_root:
        candidates.insert(0, Path(env_root) / "mocktests" / "r2-001" / "data")
    for path in candidates:
        if (path / "gen_r2_001.py").is_file():
            return path.resolve()
    raise FileNotFoundError("cannot find mocktests/r2-001/data; run from problems/ or set USAAIO_BOOK_ROOT")


DATA_DIR = find_data_dir()
_spec = importlib.util.spec_from_file_location("r2_001_data", DATA_DIR / "r2_001_data.py")
r2_001_data = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(r2_001_data)
assert r2_001_data.SEED == SEED

x_lab, y_lab = r2_001_data.train_rows("texture")
x_pool = r2_001_data.unlabelled_rows("texture")
x_val, y_val = r2_001_data.val_rows("texture")
print(x_lab.shape, x_pool.shape, x_val.shape, torch.bincount(y_lab).tolist())
print("baseline recipe:", r2_001_data.BASELINE_RECIPES["texture"])

## Budget and the committed baseline

In [ ]:
budget = r2_001_data.StepBudget(1500)
baseline = r2_001_data.baseline_score("texture")
print("baseline under the protocol:", baseline)

seam = set()


def record_seam(raw_rows):
    """Seam: record the split of every clean raw patch whose features enter a training forward."""
    for row in raw_rows:
        seam.add(r2_001_data.split_of("texture", row))

## Approach

In [ ]:
HEAD_STEPS, ROUNDS, CONF = 150, 3, 0.8

_k = torch.fft.fftfreq(10, dtype=torch.float64) * 10
_KX, _KY = torch.meshgrid(_k, _k, indexing="ij")
_RADIUS = (_KX ** 2 + _KY ** 2).sqrt()
_BANDS = [(0, 1.5), (1.5, 2.5), (2.5, 3.5), (3.5, 4.5), (4.5, 8)]


def texture_features(x):
    """(N, 1, 10, 10) patches -> (N, 23) label-free features (closed form, no training)."""
    z = x[:, 0].double()
    m, s = z.mean((1, 2)), z.std((1, 2))
    zc = (z - m[:, None, None]) / (s[:, None, None] + 1e-6)
    skew, kurt = (zc ** 3).mean((1, 2)), (zc ** 4).mean((1, 2))
    power = torch.fft.fft2(zc).abs() ** 2
    power[:, 0, 0] = 0
    flat = power.reshape(len(z), -1)
    total = flat.sum(1)
    top = flat.sort(dim=1, descending=True).values[:, :8] / total[:, None]   # spectral peakiness: stripes 2 peaks, checker 4
    bands = [(power * ((_RADIUS >= lo) & (_RADIUS < hi))).sum((1, 2)) / total for lo, hi in _BANDS]
    c_v = (zc[:, 1:, :] * zc[:, :-1, :]).mean((1, 2))
    c_h = (zc[:, :, 1:] * zc[:, :, :-1]).mean((1, 2))
    c_d = (zc[:, 1:, 1:] * zc[:, :-1, :-1]).mean((1, 2))
    c_a = (zc[:, 1:, :-1] * zc[:, :-1, 1:]).mean((1, 2))
    return torch.stack([m, s, skew, kurt, *top.T, *bands, c_v, c_h, c_d, c_a,
                        (c_v - c_h).abs(), (c_d - c_a).abs()], dim=1)


state = {}


def train_head(feats, labels, raw_rows):
    record_seam(raw_rows)                         # first (and only) training call of this head
    torch.manual_seed(SEED)
    head = nn.Linear(feats.shape[1], 3)
    opt = budget.wrap(torch.optim.Adam(head.parameters(), lr=0.05, weight_decay=1e-2))
    for _ in range(HEAD_STEPS):
        opt.zero_grad(set_to_none=True)
        F.cross_entropy(head(feats), labels).backward()
        opt.step()
    return head.eval()


def standardize(f):
    return ((f - state["mu"]) / state["sd"]).float()


def fit(labelled_x, labelled_y, pool_x, rounds=ROUNDS):
    raw = texture_features(torch.cat([labelled_x, pool_x]))
    state["mu"], state["sd"] = raw.mean(0), raw.std(0)          # statistics from labelled + unlabelled only
    f_lab, f_pool = standardize(texture_features(labelled_x)), standardize(texture_features(pool_x))
    head = train_head(f_lab, labelled_y, labelled_x)
    history = []
    for _ in range(rounds):
        with torch.no_grad():
            conf, pseudo = F.softmax(head(f_pool), dim=1).max(dim=1)
        keep = conf > CONF
        history.append((int(keep.sum()), torch.bincount(pseudo[keep], minlength=3).tolist()))
        head = train_head(torch.cat([f_lab, f_pool[keep]]), torch.cat([labelled_y, pseudo[keep]]),
                          torch.cat([labelled_x, pool_x[keep]]))
    state["head"] = head
    return history


def predict(x):
    with torch.no_grad():
        return state["head"](standardize(texture_features(x))).argmax(dim=1).to(torch.int64)


def val_acc(pred):
    return float(r2_001_data.accuracy_per_example(pred, y_val).mean())

## Validation evidence, including ablations (every choice is made here)

In [ ]:
# Ablation 1: same features and head, labelled rows only (no unlabelled pool).
fit(x_lab, y_lab, x_pool, rounds=0)
acc_no_pool = val_acc(predict(x_val))

# Alternative 2: the raw-pixel CNN of the baseline recipe, labelled only.
record_seam(x_lab)
torch.manual_seed(SEED)
cnn = r2_001_data.TextureCNN()
cnn_opt = budget.wrap(torch.optim.Adam(cnn.parameters(), lr=0.01))
for _ in range(150):
    cnn_opt.zero_grad(set_to_none=True)
    F.cross_entropy(cnn(x_lab), y_lab).backward()
    cnn_opt.step()
with torch.no_grad():
    acc_cnn = val_acc(cnn(x_val).argmax(dim=1))

# Chosen method: features + head + 3 self-training rounds on the pool.
history = fit(x_lab, y_lab, x_pool, rounds=ROUNDS)
acc_self = val_acc(predict(x_val))

print(f"val accuracy, raw-pixel CNN on 40 labels (baseline recipe):  {acc_cnn:.3f}")
print(f"val accuracy, features + logistic head, labelled only:      {acc_no_pool:.3f}")
print(f"val accuracy, features + head + {ROUNDS} self-training rounds:   {acc_self:.3f}")
for r, (kept, counts) in enumerate(history):
    print(f"  round {r}: {kept} pool patches with confidence > {CONF}, pseudo-label counts {counts}")
assert acc_self >= acc_no_pool >= acc_cnn
assert seam <= {"train", "unlabelled"} and "val" not in seam and "test" not in seam
print("seam:", seam, "| steps used before the test call:", budget.used, "of", budget.max_steps)

## The single locked-test call

In [ ]:
assert r2_001_data.test_call_count("texture") == 0
result = r2_001_data.final_test_score(predict, n_boot=1000, alpha=0.05, task="texture")
assert r2_001_data.test_call_count("texture") == 1
elapsed = budget.elapsed()
print(result)
print(f"budget: {budget.used} of {budget.max_steps} steps, elapsed {elapsed:.1f} s")

## Writeup

**Approach.** The three classes differ in structure that survives heavy pixel noise when it is pooled over the whole patch: a grating concentrates its Fourier power in two conjugate peaks, a product of perpendicular gratings in four, and blurred dots spread power over low frequencies and make the pixel histogram right-skewed on a darker background.
I compute 23 closed-form, label-free features per patch — mean, s.d., skewness, kurtosis of the pixels; the eight largest normalized power-spectrum entries of the standardized patch (spectral peakiness); the fraction of power in five radial bands (rotation-invariant); and four nearest-neighbour correlations plus two anisotropy contrasts — standardize them with statistics from the 40 labelled plus 760 unlabelled patches (never validation or test), and train a multinomial logistic-regression head (150 Adam steps, lr 0.05, weight decay 0.01).
The pool is then used by three rounds of self-training: pseudo-label the pool, keep patches with softmax confidence above 0.8, and retrain the head from scratch on labelled + kept pool patches.
With 40 labels a 72-parameter head on informative invariant features is far less data-hungry than a CNN that must learn orientation-invariance from pixels.

**Alternatives considered** (validation accuracy on the 40 validation patches, 0.025 per patch).
(1) *The raw-pixel TextureCNN on the 40 labels* (baseline recipe): 0.500 — it overfits the 40 noisy patches.
(2) *Ablation: the same features and head without the unlabelled pool*: 0.950, against 1.000 with self-training — the pool adds about two validation patches; this difference is small relative to the validation granularity, so the main gain comes from the features, and the self-training rounds were kept because they never lowered validation accuracy and the pseudo-label class counts stay near balance (212/251/250 → 240/252/250 confident patches), which argues against confirmation bias.

**Evaluation.** Choices were made on the 40 validation patches only; the locked test set was scored once with `final_test_score` (n = 400, 1,000 bootstrap resamples, α = 0.05).
Test accuracy **0.9975**, 95% bootstrap interval [0.9925, 1.0000]; the committed labelled-only CNN baseline under the same protocol scores 0.5550 [0.5050, 0.6025].
Budget: 900 of 1,500 optimizer steps (150 labelled-only head, 150 CNN alternative, 4 × 150 for the chosen head and its self-training rounds), about 2 s by `budget.elapsed()` on this CPU.

**Limitations.** The features are hand-designed for these three generators: a stripe period near the 10-pixel patch size, or a checker whose two gratings have very different contrasts, would look like the wrong class in the spectral summary, and the dot features rely on the darker background.
Self-training can also lock in a systematic error of the first head (the confident set grew mostly in class 0, 212 → 240).
I would test both by holding out the confident pseudo-labels per round and checking their agreement with a differently built classifier (for example the CNN trained on the pseudo-labelled pool), and, if more labelled data became available, by binning errors by grating period and contrast.

### Answer check

In [ ]:
assert r2_001_data.test_call_count("texture") == 1, "final_test_score must be called exactly once"
assert seam <= {"train", "unlabelled"}
assert budget.used <= 1500 and elapsed < 60
assert result.n_examples == 400 and result.ci_low <= result.score <= result.ci_high
assert result.score > baseline.score
assert result.score >= 0.8869, "full-credit tier: accuracy >= R - 0.25 g"
print(f"Problem 4 test accuracy: {result.score:.4f} (95% CI {result.ci_low:.4f}-{result.ci_high:.4f}, n = {result.n_examples}); "
      f"baseline {baseline.score:.4f}")